# Section 1.b — Logistic Regression: Separable vs Non-Separable Data

Two experiments with different class separation distances:
- **Experiment 1**: Linearly separable data (`cluster_std=1.0`)
- **Experiment 2**: Non-linearly separable data (`cluster_std=4.0`)

## 0. Setup (Google Colab only)

In [ ]:
import os, sys

if 'google.colab' in sys.modules:
    if not os.path.exists('logistic-regression'):
        os.system('git clone https://github.com/DaniloDuque/logistic-regression.git')
    os.chdir('logistic-regression')

sys.path.insert(0, os.path.abspath('src'))

## 1. Imports

In [ ]:
import torch
import numpy as np

from data_generator import generate_data
from logistic_regression import LogisticRegression
from trainer import train_with_history, compute_mae
from metrics import run_experiment, print_mae_table, print_runs_table
from visualization import plot_results

STEPS = 1000
ALPHA = 0.1

## 2. Experiment 1 — Linearly separable data

Two classes generated with `cluster_std=1.0` (compact, well-separated clusters).
The model is expected to converge quickly and achieve a low MAE.

In [ ]:
X_train_s, X_test_s, y_train_s, y_test_s = generate_data(separable=True, n_samples=500, random_state=42)

model_s = LogisticRegression(torch.zeros(X_train_s.shape[1], 1))
errors_s = train_with_history(model_s, X_train_s, y_train_s, steps=STEPS, alpha=ALPHA)

mae_train_s = compute_mae(y_train_s, model_s.forward(X_train_s))
mae_test_s  = compute_mae(y_test_s,  model_s.forward(X_test_s))

print(f"Iterations: {STEPS}")
print_mae_table('Linearly separable', mae_train_s, mae_test_s)

plot_results(model_s, X_train_s, y_train_s, errors_s, 'Experiment 1 — Linearly separable data')

## 3. Experiment 2 — Non-linearly separable data

Two classes generated with `cluster_std=4.0` (overlapping clusters).
The model is expected to struggle and achieve a higher MAE.

In [ ]:
X_train_ns, X_test_ns, y_train_ns, y_test_ns = generate_data(separable=False, n_samples=500, random_state=42)

model_ns = LogisticRegression(torch.zeros(X_train_ns.shape[1], 1))
errors_ns = train_with_history(model_ns, X_train_ns, y_train_ns, steps=STEPS, alpha=ALPHA)

mae_train_ns = compute_mae(y_train_ns, model_ns.forward(X_train_ns))
mae_test_ns  = compute_mae(y_test_ns,  model_ns.forward(X_test_ns))

print(f"Iterations: {STEPS}")
print_mae_table('Non-linearly separable', mae_train_ns, mae_test_ns)

plot_results(model_ns, X_train_ns, y_train_ns, errors_ns, 'Experiment 2 — Non-linearly separable data')

## 4. Comparative MAE table

In [ ]:
print(f"{'Case':<30} {'MAE Train':>10} {'MAE Test':>10}")
print("-" * 55)
print(f"{'Linearly separable':<30} {mae_train_s:>10.4f} {mae_test_s:>10.4f}")
print(f"{'Non-linearly separable':<30} {mae_train_ns:>10.4f} {mae_test_ns:>10.4f}")

## 5. 10 runs — Mean MAE and standard deviation

In [ ]:
maes_s  = run_experiment(separable=True,  steps=STEPS, alpha=ALPHA)
maes_ns = run_experiment(separable=False, steps=STEPS, alpha=ALPHA)

print_runs_table(maes_s, maes_ns)